In [6]:
import pandas as pd
# load dataset
df = pd.read_csv(r'C:\Users\satis\car-price-predictor\backend\data\cars_24_combined.csv')

# drop redundant column
df = df.drop(columns=['Unnamed: 0'])

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8015 entries, 0 to 8014
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Car Name  8014 non-null   str    
 1   Year      8014 non-null   float64
 2   Distance  8015 non-null   int64  
 3   Owner     8015 non-null   int64  
 4   Fuel      8015 non-null   str    
 5   Location  7802 non-null   str    
 6   Drive     8015 non-null   str    
 7   Type      8015 non-null   str    
 8   Price     8015 non-null   int64  
dtypes: float64(1), int64(3), str(5)
memory usage: 563.7 KB


In [8]:
df.head()

,Car Name,Year,Distance,Owner,Fuel,Location,Drive,Type,Price
0,Maruti S PRESSO,2022.0,3878,1,PETROL,HR-98,Manual,HatchBack,514000
1,Hyundai Xcent,2018.0,32041,1,PETROL,TN-22,Manual,Sedan,674000
2,Tata Safari,2021.0,96339,1,DIESEL,TS-08,Automatic,SUV,1952000
3,Maruti Vitara Brezza,2019.0,51718,1,DIESEL,WB-24,Manual,SUV,690000
4,Tata Tiago,2021.0,19811,1,PETROL,HR-51,Manual,HatchBack,526000


In [9]:
# shape of the data
print(f"Data Shape :{df.shape}")

Data Shape :(8015, 9)


In [10]:
# missing values
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
Car Name      1
Year          1
Distance      0
Owner         0
Fuel          0
Location    213
Drive         0
Type          0
Price         0
dtype: int64


In [11]:
df = df.dropna(subset=['Car Name','Year'])

most_freq_locations = df['Location'].mode()[0]
df['Location'] = df['Location'].fillna(most_freq_locations)

# Create 'Car_age' from 'Year'
df['Car_age'] = 2024 - df['Year']

# Drop the Year column
df = df.drop(columns=['Year'])

print("Missing Values after cleaning")
print(df.isnull().sum())

display(df.head(3))

Missing Values after cleaning
Car Name    0
Distance    0
Owner       0
Fuel        0
Location    0
Drive       0
Type        0
Price       0
Car_age     0
dtype: int64


,Car Name,Distance,Owner,Fuel,Location,Drive,Type,Price,Car_age
0,Maruti S PRESSO,3878,1,PETROL,HR-98,Manual,HatchBack,514000,2.0
1,Hyundai Xcent,32041,1,PETROL,TN-22,Manual,Sedan,674000,6.0
2,Tata Safari,96339,1,DIESEL,TS-08,Automatic,SUV,1952000,3.0


In [12]:
import numpy as np
# extract brand
df['Brand'] = df['Car Name'].apply(lambda x: str(x).split(' ')[0])
# extract Model from the car name
df['Model'] = df['Car Name'].apply(lambda x: ' '.join(str(x).split(' ')[1:]))

# drop old car name column
df = df.drop(columns=['Car Name'])

# log transformation for the price(target variable)
df['Price'] = np.log1p(df['Price'])

print(df.head())


   Distance  Owner    Fuel Location      Drive       Type      Price  Car_age  \
0      3878      1  PETROL    HR-98     Manual  HatchBack  13.149980      2.0   
1     32041      1  PETROL    TN-22     Manual      Sedan  13.420987      6.0   
2     96339      1  DIESEL    TS-08  Automatic        SUV  14.484366      3.0   
3     51718      1  DIESEL    WB-24     Manual        SUV  13.444448      5.0   
4     19811      1  PETROL    HR-51     Manual  HatchBack  13.173058      3.0   

     Brand          Model  
0   Maruti       S PRESSO  
1  Hyundai          Xcent  
2     Tata         Safari  
3   Maruti  Vitara Brezza  
4     Tata          Tiago  


In [13]:
luxury_brands = ['Mercedes-Benz', 'BMW', 'Audi', 'Lexus', 'Porsche', 'Jaguar', 'Land Rover', 'Maserati', 'Bentley', 'Rolls-Royce']

df = df[~df['Brand'].isin(luxury_brands)]

print(f"Dataset size after scoping to commuter vehicles: {len(df)} rows")

Dataset size after scoping to commuter vehicles: 8013 rows


In [14]:
df['Age_x_Distance'] = df['Car_age'] * df['Distance']
# Group rare models into an 'Other' bucket to prevent structural noise
model_counts = df['Model'].value_counts()
rare_models = model_counts[model_counts < 5].index
df['Model'] = df['Model'].replace(rare_models, 'Other')

In [15]:
# 1. The RTO State Dictionary
state_mapping = {
    'AP': 'Andhra Pradesh', 'AR': 'Arunachal Pradesh', 'AS': 'Assam',
    'BR': 'Bihar', 'CG': 'Chhattisgarh', 'CH': 'Chandigarh',
    'DD': 'Daman and Diu', 'DL': 'Delhi', 'DN': 'Dadra and Nagar Haveli',
    'GA': 'Goa', 'GJ': 'Gujarat', 'HR': 'Haryana', 'HP': 'Himachal Pradesh',
    'JH': 'Jharkhand', 'JK': 'Jammu and Kashmir', 'KA': 'Karnataka',
    'KL': 'Kerala', 'LD': 'Lakshadweep', 'MH': 'Maharashtra',
    'ML': 'Meghalaya', 'MN': 'Manipur', 'MP': 'Madhya Pradesh',
    'MZ': 'Mizoram', 'NL': 'Nagaland', 'OD': 'Odisha', 'PB': 'Punjab',
    'PY': 'Puducherry', 'RJ': 'Rajasthan', 'SK': 'Sikkim',
    'TN': 'Tamil Nadu', 'TR': ' त्रिपुरा', 'TS': 'Telangana',
    'UK': 'Uttarakhand', 'UP': 'Uttar Pradesh', 'WB': 'West Bengal'
}

# 2. Extract the first two letters of the Location code and map it
df['State'] = df['Location'].astype(str).str[:2].str.upper().map(state_mapping)

# 3. Handle any weird/missing codes by labeling them 'Other'
df['State'] = df['State'].fillna('Other')

# 4. Drop the old noisy Location column
df = df.drop(columns=['Location'])

print(df['State'].value_counts())

State
Maharashtra       1639
Karnataka         1215
Delhi              858
Gujarat            764
Tamil Nadu         661
Telangana          653
Haryana            602
Uttar Pradesh      462
West Bengal        280
Punjab             186
Rajasthan          151
Kerala             142
Madhya Pradesh     134
Bihar              117
Andhra Pradesh      83
Chandigarh          65
Other                1
Name: count, dtype: int64


In [16]:
df.head()


,Distance,Owner,Fuel,Drive,Type,Price,Car_age,Brand,Model,Age_x_Distance,State
0,3878,1,PETROL,Manual,HatchBack,13.149980,2.0,Maruti,S PRESSO,7756.0,Haryana
1,32041,1,PETROL,Manual,Sedan,13.420987,6.0,Hyundai,Xcent,192246.0,Tamil Nadu
2,96339,1,DIESEL,Automatic,SUV,14.484366,3.0,Tata,Safari,289017.0,Telangana
3,51718,1,DIESEL,Manual,SUV,13.444448,5.0,Maruti,Vitara Brezza,258590.0,West Bengal
4,19811,1,PETROL,Manual,HatchBack,13.173058,3.0,Tata,Tiago,59433.0,Haryana


In [17]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

# seperate features from target
X = df.drop(columns=['Price'])
y = df['Price']

# add brand and model to the catboost text list
categorical_features = ['Model','Brand','State','Fuel','Drive','Type']

# split the data
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# initialize and train
model = CatBoostRegressor(
    iterations=800,
    learning_rate=0.08,
    depth=8,
    l2_leaf_reg=5,
    cat_features=categorical_features,
    verbose=False
)

model.fit(X_train,y_train)

CatBoostRegressor(cat_features=['Model', 'Brand', 'State', 'Fuel', 'Drive', 'Type'], depth=8, iterations=800, l2_leaf_reg=5, learning_rate=0.08, loss_function='RMSE', verbose=False)

In [18]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Ask CatBoost to predict the LOG prices
log_predictions = model.predict(X_test)

# 2. Reverse the math: Convert logs back to real Rupees
real_predictions = np.expm1(log_predictions)
real_y_test = np.expm1(y_test)

# 3. Calculate the true Rupee errors
real_mae = mean_absolute_error(real_y_test, real_predictions)
real_rmse = np.sqrt(mean_squared_error(real_y_test, real_predictions))

# R² stays the same (calculated on the log values to judge pure variance)
r2 = r2_score(y_test, log_predictions) 

# 4. Print the honest results
print("--- Evaluation Metrics ---")
print(f"True Mean Absolute Error (MAE): ₹{real_mae:,.2f}")
print(f"True Root Mean Squared Error (RMSE): ₹{real_rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

--- Evaluation Metrics ---
True Mean Absolute Error (MAE): ₹61,024.72
True Root Mean Squared Error (RMSE): ₹99,395.67
R² Score: 0.8831


In [19]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# ==========================================
# 1. GENERATE THE BASELINE (DUMMY) MODEL
# ==========================================
# The dummy regressor always predicts the mean of y_train
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)

# Predict and reverse log transform
baseline_log_preds = baseline_model.predict(X_test)
baseline_real_preds = np.expm1(baseline_log_preds)

baseline_mae = mean_absolute_error(real_y_test, baseline_real_preds)
baseline_r2 = r2_score(y_test, baseline_log_preds)

print("--- BASELINE (FLOOR) METRICS ---")
print(f"Baseline Guessing Strategy MAE: ₹{baseline_mae:,.2f}")
print(f"Baseline R² Score: {baseline_r2:.4f}\n")

print(f"Your optimized CatBoost model vs Baseline improvement margin: ₹{baseline_mae - real_mae:,.2f} savings per prediction.") 

--- BASELINE (FLOOR) METRICS ---
Baseline Guessing Strategy MAE: ₹188,514.49
Baseline R² Score: -0.0013

Your optimized CatBoost model vs Baseline improvement margin: ₹127,489.77 savings per prediction.


In [21]:
model.save_model(r"C:\Users\satis\car-price-predictor\backend\models\catboost_engine.cbm")

In [22]:
import json

brand_to_models = df.groupby('Brand')['Model'].unique().apply(list).to_dict()

metadata = {
    "brands_and_models": brand_to_models,
    "states": df['State'].unique().tolist(), # Updated to States
    "fuel_types": df['Fuel'].unique().tolist(),
    "transmissions": df['Drive'].unique().tolist(),
    "body_types": df['Type'].unique().tolist()
}

with open(r'C:\Users\satis\car-price-predictor\backend\models\dropdown_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)